# 02 - NumPy Operations: Vectorization, Broadcasting, Masking

**Prerequisite:** notebook 01.
**This notebook:** the operations you will use every day - element-wise math (ufuncs), broadcasting, boolean masks, fancy indexing, sorting and searching, and aggregations.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## 1. ufuncs - element-wise functions

A **ufunc** (universal function) applies an operation to every element, in C, without a Python loop. Every arithmetic operator on an array is a ufunc call.

Java 8 analogy: `IntStream.of(a).map(x -> x * 2).toArray()` - except the mapping runs in compiled C and you do not write the lambda.

In [ ]:
a = np.array([1, 2, 3, 4], dtype=float)
b = np.array([10, 20, 30, 40], dtype=float)

print("a + b  :", a + b)          # np.add
print("a * b  :", a * b)          # np.multiply  (element-wise, NOT matrix multiply)
print("b / a  :", b / a)
print("b // a :", b // a)         # floor division
print("a ** 2 :", a ** 2)
print("b % 3  :", b % 3)
print("-a     :", -a)

In [ ]:
print("sqrt   :", np.sqrt(a))
print("exp    :", np.exp(a).round(3))
print("log    :", np.log(b).round(3))
print("sin    :", np.sin(a).round(3))
print("abs    :", np.abs([-1, 2, -3]))
print("clip   :", np.clip(np.array([-5, 0, 5, 15]), 0, 10))   # like Math.min(Math.max(x,lo),hi)
print("sign   :", np.sign([-3, 0, 3]))
print("maximum:", np.maximum([1, 5, 3], [4, 2, 3]))           # element-wise max of TWO arrays

> `np.max(x)` = the largest value **inside** x. `np.maximum(x, y)` = element-wise pairwise max of **two** arrays. Same pattern for `min`/`minimum`. Mixing these up is a classic bug.

### Matrix multiply vs element-wise multiply

In [ ]:
m = np.array([[1, 2], [3, 4]])
n = np.array([[5, 6], [7, 8]])

print("m * n  (element-wise):\n", m * n)
print()
print("m @ n  (matrix product):\n", m @ n)
print()
print("np.dot(m, n) is the same as @:\n", np.dot(m, n))

### Writing into an existing array with `out=`

Every ufunc accepts `out=` to write the result into a pre-allocated array instead of allocating a new one. This matters when arrays are large.

In [ ]:
big = np.arange(6, dtype=float)
target = np.empty_like(big)
np.multiply(big, 2, out=target)
print(target)

# In-place operators do the same thing and skip the allocation
big *= 2
print(big)

## 2. Broadcasting - the rule that explains everything

When two arrays have different shapes, NumPy tries to **stretch** them to a common shape without copying data.

**The algorithm:** line the shapes up from the **right**. For each pair of dimensions:
1. equal -> fine
2. one of them is 1 -> stretch that one
3. otherwise -> `ValueError`

A missing leading dimension counts as 1.

In [ ]:
# Case 1: array + scalar. Scalar is shape () -> stretched to everything.
a = np.array([1, 2, 3])
print(a + 10)

In [ ]:
# Case 2: (3,4) + (4,)  ->  the (4,) is treated as (1,4) and repeated down the rows
m = np.zeros((3, 4))
row = np.array([1, 2, 3, 4])
print((m + row))
print("shapes:", m.shape, "+", row.shape, "->", (m + row).shape)

In [ ]:
# Case 3: (3,1) + (1,4) -> BOTH stretch, giving (3,4). This is an outer operation.
col = np.array([[10], [20], [30]])      # shape (3,1)
row = np.array([[1, 2, 3, 4]])          # shape (1,4)
print(col + row)
print("shapes:", col.shape, "+", row.shape, "->", (col + row).shape)

In [ ]:
# Case 4: incompatible
try:
    np.zeros((3, 4)) + np.zeros((3,))
except ValueError as e:
    print("ValueError:", e)
print()
print("Fix: make the intent explicit with a new axis ->", (np.zeros((3, 4)) + np.zeros((3,))[:, None]).shape)

### A real use: normalising columns

Subtracting the column means from a matrix is one line, no loops.

In [ ]:
data = rng.normal(loc=[100, 5, 0.2], scale=[10, 1, 0.05], size=(6, 3))
print("column means :", data.mean(axis=0).round(3))
print("column stds  :", data.std(axis=0).round(3))

standardised = (data - data.mean(axis=0)) / data.std(axis=0)
print()
print("after standardising, means are ~0:", standardised.mean(axis=0).round(10))
print("and stds are ~1                  :", standardised.std(axis=0).round(10))

### `keepdims=True`

A reduction removes an axis, which then breaks broadcasting for row-wise work. `keepdims=True` keeps it as a length-1 axis so it broadcasts back cleanly.

In [ ]:
m = np.arange(12).reshape(3, 4).astype(float)

print("row sums            :", m.sum(axis=1).shape)
print("row sums keepdims   :", m.sum(axis=1, keepdims=True).shape)
print()
print("each row divided by its own sum:\n", (m / m.sum(axis=1, keepdims=True)).round(3))

## 3. Boolean masking - this is your `WHERE` clause

A comparison on an array produces a **boolean array** of the same shape. Indexing with that boolean array selects the `True` positions.

In [ ]:
x = np.array([5, 12, 3, 18, 7, 20])

mask = x > 10
print("mask :", mask, " <- this is the WHERE predicate")
print("x[mask]:", x[mask], " <- SELECT * FROM x WHERE value > 10")

In [ ]:
# Combining conditions: use & | ~ with PARENTHESES. Python's and/or/not do not work here.
print("10 < x < 19 :", x[(x > 10) & (x < 19)])
print("x<5 or x>18 :", x[(x < 5) | (x > 18)])
print("NOT > 10    :", x[~(x > 10)])

try:
    x[(x > 10) and (x < 19)]
except ValueError as e:
    print()
    print("using 'and' raises:", str(e)[:70], "...")

**Why?** `and` calls `bool()` on the whole array, and an array with more than one element has no single truth value. `&` is element-wise. Because `&` binds tighter than `>` in Python, the parentheses are mandatory.

In [ ]:
# Useful boolean reductions
print("any > 18        :", (x > 18).any())     # EXISTS
print("all > 0         :", (x > 0).all())      # NOT EXISTS (counter-example)
print("count > 10      :", (x > 10).sum())     # COUNT(*) WHERE ... - True counts as 1
print("fraction > 10   :", (x > 10).mean())    # share of rows matching

In [ ]:
# Assignment through a mask - this is an UPDATE ... WHERE
y = x.copy()
y[y > 15] = 15                 # cap values
print("capped:", y)

# np.where = SQL CASE WHEN ... THEN ... ELSE ... END
print("where :", np.where(x > 10, "high", "low"))
print("where :", np.where(x > 10, x * 100, x))

In [ ]:
# np.select = a multi-branch CASE WHEN
conditions = [x < 5, x < 15, x >= 15]
choices    = ["small", "medium", "large"]
print(np.select(conditions, choices, default="unknown"))

## 4. Fancy (integer-array) indexing

Index with a list/array of positions. The result has the shape of the **index array**, and it is always a copy.

In [ ]:
x = np.array([10, 20, 30, 40, 50])
idx = [4, 0, 0, 2]
print("x[idx]  :", x[idx], " <- repeats and reordering are allowed")
print("shape follows the index:", x[np.array([[0, 1], [2, 3]])])

In [ ]:
m = np.arange(1, 13).reshape(3, 4)
print(m)
print()
# Pairs of coordinates: rows [0,2], cols [1,3] -> elements (0,1) and (2,3)
print("m[[0, 2], [1, 3]] =", m[[0, 2], [1, 3]])
print()
# Whole rows in a given order
print("m[[2, 0]] =\n", m[[2, 0]])
print()
# np.ix_ builds the cross product (all selected rows x all selected cols)
print("m[np.ix_([0, 2], [1, 3])] =\n", m[np.ix_([0, 2], [1, 3])])

## 5. Aggregations

| NumPy | SQL |
|---|---|
| `arr.sum()` | `SUM()` |
| `arr.mean()` | `AVG()` |
| `arr.min()` / `arr.max()` | `MIN()` / `MAX()` |
| `arr.std()`, `arr.var()` | `STDDEV()`, `VARIANCE()` |
| `np.median(arr)` | `PERCENTILE_CONT(0.5)` |
| `np.percentile(arr, 90)` | `PERCENTILE_CONT(0.9)` |
| `arr.argmin()` / `arr.argmax()` | position of the min/max |
| `np.unique(arr)` | `SELECT DISTINCT` |
| `arr.cumsum()` | `SUM() OVER (ORDER BY ...)` |

In [ ]:
v = np.array([4, 8, 15, 16, 23, 42])
print("sum      :", v.sum())
print("mean     :", v.mean())
print("median   :", np.median(v))
print("std (pop):", v.std().round(3))          # NumPy divides by N by default
print("std (sam):", v.std(ddof=1).round(3))    # ddof=1 divides by N-1, what SQL STDDEV_SAMP does
print("p25, p90 :", np.percentile(v, [25, 90]))
print("argmax   :", v.argmax(), "-> value", v[v.argmax()])
print("cumsum   :", v.cumsum())
print("diff     :", np.diff(v), " <- consecutive differences, like LAG()")

### Missing values: `nan` poisons everything

`np.nan` is a float. Any arithmetic involving it returns `nan`, so a single missing value ruins a whole aggregate. Use the `nan*` variants, which skip it - exactly like SQL aggregates ignoring `NULL`.

In [ ]:
w = np.array([1.0, 2.0, np.nan, 4.0])
print("w.sum()      :", w.sum(), "  <- one nan poisons the result")
print("np.nansum(w) :", np.nansum(w), " <- skips it, like SQL SUM ignoring NULL")
print("np.nanmean(w):", np.nanmean(w))
print()
print("nan == nan   :", np.nan == np.nan, "  <- never compare with ==, same trap as SQL NULL = NULL")
print("np.isnan(w)  :", np.isnan(w), " <- this is IS NULL")
print("drop nans    :", w[~np.isnan(w)])

## 6. Sorting, searching, set operations

In [ ]:
u = np.array([30, 10, 20, 50, 40])
print("np.sort  (copy)   :", np.sort(u))
print("argsort (positions):", np.argsort(u), " <- the ORDER BY index")
print("descending         :", np.sort(u)[::-1])

In [ ]:
# argsort is how you sort one array BY another - like ORDER BY a different column
names  = np.array(["kolkata", "pune", "delhi", "chennai"])
scores = np.array([72, 91, 65, 88])

order = np.argsort(-scores)          # negative for descending
print("ranked:", list(zip(names[order].tolist(), scores[order].tolist())))

In [ ]:
s = np.array([10, 20, 30, 40, 50])
print("searchsorted(35) :", np.searchsorted(s, 35), " <- insertion point in a sorted array (binary search)")
print("bucket 5,25,55   :", np.searchsorted(s, [5, 25, 55]))
print()
print("np.unique        :", np.unique([3, 1, 2, 3, 1]))
vals, counts = np.unique([3, 1, 2, 3, 1], return_counts=True)
print("value counts     :", dict(zip(vals.tolist(), counts.tolist())), " <- GROUP BY + COUNT(*)")

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([3, 4, 5, 6])
print("intersect1d :", np.intersect1d(a, b), " <- INNER JOIN / INTERSECT")
print("union1d     :", np.union1d(a, b),     " <- UNION")
print("setdiff1d   :", np.setdiff1d(a, b),   " <- EXCEPT / LEFT JOIN ... WHERE b IS NULL")
print("isin        :", np.isin(a, b),        " <- WHERE x IN (...)")
print("a[isin]     :", a[np.isin(a, b)])

## 7. A worked mini-problem

Sensor readings, 1000 samples across 4 sensors. Find, per sensor: the mean of valid readings, the count of outliers (more than 3 standard deviations from that sensor's mean), and the index of the maximum reading. No loops.

In [ ]:
rng = np.random.default_rng(7)
readings = rng.normal(loc=50, scale=5, size=(1000, 4))

# inject some missing values and some spikes
readings[rng.integers(0, 1000, 20), rng.integers(0, 4, 20)] = np.nan
readings[rng.integers(0, 1000, 10), rng.integers(0, 4, 10)] = 200.0

means = np.nanmean(readings, axis=0)
stds  = np.nanstd(readings, axis=0)

z = np.abs(readings - means) / stds              # broadcasting: (1000,4) against (4,)
outliers = (z > 3)                               # nan comparisons are False, so nans are not counted

print("per-sensor mean       :", means.round(2))
print("per-sensor nan count  :", np.isnan(readings).sum(axis=0))
print("per-sensor outliers   :", outliers.sum(axis=0))
print("row index of each max :", np.nanargmax(readings, axis=0))

In [ ]:
# Clean version: replace outliers and nans with the sensor mean
cleaned = np.where(np.isnan(readings) | outliers, means, readings)
print("cleaned means:", cleaned.mean(axis=0).round(2))
print("any nan left? ", np.isnan(cleaned).any())

## 8. Exercises

**E1.** `a = np.arange(1, 21)`. Select all values divisible by 3 **or** greater than 15, using one boolean expression.

**E2.** Build a 10x10 multiplication table using broadcasting only (no loops, no `np.outer`).

**E3.** `prices = np.array([100., 102., 99., 105., 103.])`. Compute the daily percentage change. (Hint: `np.diff` plus a slice.)

**E4.** `m = rng.integers(0, 100, (5, 3))`. Return the rows sorted by the value in column 1, descending.

**E5.** Why does `np.zeros((4,3)) + np.zeros((4,))` fail while `np.zeros((3,4)) + np.zeros((4,))` works? Answer in one sentence.

In [ ]:
# Solutions
a = np.arange(1, 21)
print("E1:", a[(a % 3 == 0) | (a > 15)])
print()

n = np.arange(1, 11)
print("E2:\n", n[:, None] * n[None, :])
print()

prices = np.array([100., 102., 99., 105., 103.])
print("E3:", (np.diff(prices) / prices[:-1] * 100).round(2), "%")
print()

rng = np.random.default_rng(1)
m = rng.integers(0, 100, (5, 3))
print("E4 original:\n", m)
print("E4 sorted by col 1 desc:\n", m[np.argsort(-m[:, 1])])
print()
print("E5: shapes align from the RIGHT. (4,3) vs (4,) compares 3 with 4 -> fail;")
print("    (3,4) vs (4,) compares 4 with 4 -> ok.")

## Recap

- Every operator on an array is a **ufunc**: element-wise, in C, no Python loop.
- **Broadcasting**: align shapes from the right; a dimension of 1 stretches; anything else must match.
- `keepdims=True` keeps a reduced axis so the result broadcasts back.
- Boolean masks are your `WHERE`. Use `& | ~` with parentheses, never `and/or/not`.
- `np.where` is `CASE WHEN`; `np.select` is a multi-branch `CASE`.
- `nan` poisons aggregates - use `nansum`, `nanmean`, `np.isnan`.
- `argsort` sorts one array by another.

**Next:** `03_numpy_advanced.ipynb` - memory layout, strides, structured arrays, datetimes, linear algebra.